# Validate HEX ZMQ Server/Client

This notebook checks the HEX ZMQ policy server without connecting to Sonic, cameras, or robot action publishers.

It loads one frame from `/media/bsh/data/hex_data/cola_0610`, builds the same HEX observation used by `/media/bsh/code/HEX/deployment/run_hex_inference.py`, calls the server, unnormalizes the returned action with HEX `dataset_statistics.json`, and inspects `motion_token` plus `hand_binary_action`.

Run this notebook with the `hex` conda environment:

```bash
source /media/bsh/miniconda3/etc/profile.d/conda.sh
conda activate hex
jupyter lab
```

Note: HEX server defaults to port `10093`. The `--policy-port 5550` in a starVLA command will not hit this HEX server unless you explicitly start HEX on `5550`.

In [1]:
from __future__ import annotations

import json
import os
import socket
import subprocess
import time
from pathlib import Path
from typing import Any

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-hex")

import cv2
import matplotlib.pyplot as plt
import msgpack_numpy as mnp
import numpy as np
import pyarrow.parquet as pq
import zmq

HEX_ROOT = Path("/media/bsh/code/HEX")
DATA_ROOT = Path("/media/bsh/data/hex_data/cola_0610")
MODEL_PATH = HEX_ROOT / "pretrained_models/hex/g1_sonic_real_world_pick_cola_2B/hex_ac100_3w_8gpu_state_query_history0_ft/checkpoints/steps_30000_pytorch_model.pt"
STATS_PATH = HEX_ROOT / "pretrained_models/hex/g1_sonic_real_world_pick_cola_2B/hex_ac100_3w_8gpu_state_query_history0_ft/dataset_statistics.json"
SERVER_SCRIPT = HEX_ROOT / "deployment/model_server/run_hex_server.py"
if not SERVER_SCRIPT.exists():
    SERVER_SCRIPT = HEX_ROOT / "deployment/run_hex_server.py"

POLICY_HOST = "127.0.0.1"
POLICY_PORT = 10093
EMBODIMENT_TAG = "unitree_g1_sonic"
PROMPT = "Walk forward, grab the cola and throw into the trash bin"

MOTION_TOKEN_DIM = 64
HAND_BINARY_DIM = 2
ACTION_DIM = MOTION_TOKEN_DIM + HAND_BINARY_DIM
HAND_BINARY_CLOSE_THRESHOLD = 0.5
CLIP_NORMALIZED_ACTIONS = False

print("HEX root:", HEX_ROOT)
print("data:", DATA_ROOT)
print("model exists:", MODEL_PATH.exists(), MODEL_PATH)
print("stats exists:", STATS_PATH.exists(), STATS_PATH)
print("server script:", SERVER_SCRIPT)

HEX root: /media/bsh/code/HEX
data: /media/bsh/data/hex_data/cola_0610
model exists: True /media/bsh/code/HEX/pretrained_models/hex/g1_sonic_real_world_pick_cola_2B/hex_ac100_3w_8gpu_state_query_history0_ft/checkpoints/steps_30000_pytorch_model.pt
stats exists: True /media/bsh/code/HEX/pretrained_models/hex/g1_sonic_real_world_pick_cola_2B/hex_ac100_3w_8gpu_state_query_history0_ft/dataset_statistics.json
server script: /media/bsh/code/HEX/deployment/run_hex_server.py


## Optional: start the HEX policy server

If the HEX server is already running in a terminal, keep `RUN_SERVER = False`. To start it from the notebook, set `RUN_SERVER = True` once. Logs go to `/tmp/hex_zmq_server_10093.log`.

In [ ]:
RUN_SERVER = False
SERVER_LOG = Path(f"/tmp/hex_zmq_server_{POLICY_PORT}.log")

def is_port_open(host: str, port: int, timeout: float = 1.0) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0

if RUN_SERVER:
    if is_port_open(POLICY_HOST, POLICY_PORT, timeout=0.5):
        print(f"Server already appears to be listening on {POLICY_HOST}:{POLICY_PORT}")
    else:
        cmd = f"""
        source /media/bsh/miniconda3/etc/profile.d/conda.sh
        conda activate hex
        cd {HEX_ROOT}
        python {SERVER_SCRIPT} \\
            --model_path {MODEL_PATH} \\
            --port {POLICY_PORT} \\
            --device cuda \\
            --dtype bfloat16
        """
        log_f = SERVER_LOG.open("ab")
        server_proc = subprocess.Popen(
            ["bash", "-lc", cmd],
            stdout=log_f,
            stderr=subprocess.STDOUT,
            cwd=str(HEX_ROOT),
            start_new_session=True,
        )
        print("Started server process:", server_proc.pid)
        print("Log:", SERVER_LOG)
        for i in range(240):
            if is_port_open(POLICY_HOST, POLICY_PORT, timeout=0.2):
                print(f"Server port is open after {i + 1}s")
                break
            if server_proc.poll() is not None:
                raise RuntimeError(f"Server exited with code {server_proc.returncode}. Check {SERVER_LOG}")
            time.sleep(1)
        else:
            raise TimeoutError(f"Server did not open {POLICY_HOST}:{POLICY_PORT}. Check {SERVER_LOG}")
else:
    print("RUN_SERVER is False; expecting an existing server or using ping to verify it is down.")

In [ ]:
class ZMQPolicyClient:
    def __init__(self, host: str = "127.0.0.1", port: int = 10093, timeout_ms: int = 120000):
        self.context = zmq.Context()
        self.host = host
        self.port = port
        self.timeout_ms = timeout_ms
        self._init_socket()

    def _init_socket(self) -> None:
        if hasattr(self, "socket"):
            self.socket.close(linger=0)
        self.socket = self.context.socket(zmq.REQ)
        self.socket.setsockopt(zmq.RCVTIMEO, self.timeout_ms)
        self.socket.setsockopt(zmq.SNDTIMEO, self.timeout_ms)
        self.socket.connect(f"tcp://{self.host}:{self.port}")

    @staticmethod
    def _to_bytes(data: Any) -> bytes:
        return mnp.packb(data, default=mnp.encode)

    @staticmethod
    def _from_bytes(data: bytes) -> Any:
        return mnp.unpackb(data, object_hook=mnp.decode, raw=False)

    def call_endpoint(self, endpoint: str, data: dict[str, Any] | None = None) -> Any:
        request: dict[str, Any] = {"endpoint": endpoint}
        if data is not None:
            request["data"] = data
        try:
            self.socket.send(self._to_bytes(request))
            message = self.socket.recv()
        except zmq.error.Again:
            self._init_socket()
            raise
        response = self._from_bytes(message)
        if isinstance(response, dict) and "error" in response:
            raise RuntimeError(f"Server error: {response['error']}")
        return response

    def ping(self):
        return self.call_endpoint("ping")

    def get_action(self, observation: dict[str, Any], options: dict[str, Any] | None = None):
        return tuple(self.call_endpoint("get_action", {"observation": observation, "options": options}))

    def close(self) -> None:
        self.socket.close(linger=0)
        self.context.term()

client = ZMQPolicyClient(POLICY_HOST, POLICY_PORT)
ping = client.ping()
ping

In [ ]:
with STATS_PATH.open("r") as f:
    all_stats = json.load(f)
stats = all_stats[EMBODIMENT_TAG]

def q99_normalize(x: np.ndarray, stat: dict[str, Any]) -> np.ndarray:
    q01 = np.asarray(stat["q01"], dtype=np.float32)
    q99 = np.asarray(stat["q99"], dtype=np.float32)
    denom = np.maximum(q99 - q01, 1e-6)
    return (2.0 * (np.asarray(x, dtype=np.float32) - q01) / denom - 1.0).astype(np.float32)

def q99_unnormalize(x: np.ndarray, stat: dict[str, Any], clip: bool = False) -> np.ndarray:
    q01 = np.asarray(stat["q01"], dtype=np.float32)
    q99 = np.asarray(stat["q99"], dtype=np.float32)
    mask = np.asarray(stat.get("mask", np.ones_like(q01, dtype=bool)), dtype=bool)
    out = np.asarray(x, dtype=np.float32).copy()
    dim = min(out.shape[-1], q01.shape[0])
    normalized = np.clip(out[..., :dim], -1.0, 1.0) if clip else out[..., :dim]
    raw = 0.5 * (normalized + 1.0) * (q99[:dim] - q01[:dim]) + q01[:dim]
    out[..., :dim] = np.where(mask[:dim], raw, out[..., :dim])
    return out.astype(np.float32, copy=False)

print("state stats dim:", len(stats["state"]["q01"]))
print("action stats dim:", len(stats["action"]["q01"]))

## Pick one offline frame

`FRAME_INDEX=None` uses the middle frame of the selected episode.

In [ ]:
EPISODE_INDEX = 0
FRAME_INDEX = None

def episode_chunk(episode_index: int) -> int:
    return episode_index // 1000

def read_episode_table(episode_index: int):
    parquet_path = DATA_ROOT / f"data/chunk-{episode_chunk(episode_index):03d}/episode_{episode_index:06d}.parquet"
    if not parquet_path.exists():
        raise FileNotFoundError(parquet_path)
    return pq.read_table(parquet_path), parquet_path

def row_value(table, name: str, row: int):
    return table[name][row].as_py()

def read_video_frame(episode_index: int, frame_index: int) -> np.ndarray:
    video_path = DATA_ROOT / f"videos/chunk-{episode_chunk(episode_index):03d}/observation.images.ego_view/episode_{episode_index:06d}.mp4"
    if not video_path.exists():
        raise FileNotFoundError(video_path)
    cap = cv2.VideoCapture(str(video_path))
    try:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, bgr = cap.read()
        if not ok or bgr is None:
            raise RuntimeError(f"Failed to read frame {frame_index} from {video_path}")
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    finally:
        cap.release()

table, parquet_path = read_episode_table(EPISODE_INDEX)
frame_index = table.num_rows // 2 if FRAME_INDEX is None else int(FRAME_INDEX)
frame_index = int(np.clip(frame_index, 0, table.num_rows - 1))

image = read_video_frame(EPISODE_INDEX, frame_index)
state_43 = np.asarray(row_value(table, "observation.state", frame_index), dtype=np.float32)
projected_gravity = np.asarray(row_value(table, "observation.projected_gravity", frame_index), dtype=np.float32)
state_46 = np.concatenate([state_43, projected_gravity]).astype(np.float32)
state_norm = q99_normalize(state_46.reshape(1, 1, -1), stats["state"])

gt_motion_token = np.asarray(row_value(table, "action.motion_token", frame_index), dtype=np.float32)
gt_hand_binary = np.asarray(row_value(table, "action.hand_binary", frame_index), dtype=np.float32)

hex_observation = {
    "batch_images": [[image]],
    "instructions": [PROMPT],
    "state": state_norm,
    "tags": [EMBODIMENT_TAG],
}

print("parquet:", parquet_path)
print("episode/frame:", EPISODE_INDEX, frame_index, "rows:", table.num_rows)
print("image:", image.shape, image.dtype, "state:", state_norm.shape, state_norm.dtype)
print("gt motion_token shape:", gt_motion_token.shape, "gt hand_binary:", gt_hand_binary)

plt.figure(figsize=(7, 5))
plt.imshow(image)
plt.axis("off")
plt.title(f"episode {EPISODE_INDEX}, frame {frame_index}")
plt.show()

## Run HEX inference and inspect outputs

In [ ]:
t0 = time.monotonic()
action_response, info = client.get_action(hex_observation, options={"do_sample": False})
dt = time.monotonic() - t0

normalized_actions = np.asarray(action_response["actions"], dtype=np.float32)
if normalized_actions.ndim == 3 and normalized_actions.shape[0] == 1:
    normalized_actions = normalized_actions[0]
if normalized_actions.ndim == 1:
    normalized_actions = normalized_actions.reshape(1, -1)

over_bound = np.abs(normalized_actions[:, :MOTION_TOKEN_DIM]) > 1.0
actions = q99_unnormalize(normalized_actions[:, :ACTION_DIM], stats["action"], clip=CLIP_NORMALIZED_ACTIONS)
motion_tokens = actions[:, :MOTION_TOKEN_DIM]
hand_binary = actions[:, MOTION_TOKEN_DIM:ACTION_DIM]
hand_closed = hand_binary >= HAND_BINARY_CLOSE_THRESHOLD

print(f"request latency: {dt:.3f}s")
print("server info:", info)
print("normalized actions shape:", normalized_actions.shape)
print("normalized motion max_abs:", float(np.abs(normalized_actions[:, :MOTION_TOKEN_DIM]).max()))
print("normalized motion over [-1,1] ratio:", float(over_bound.mean()))
print("unnormalized actions shape:", actions.shape)
print("motion_tokens shape:", motion_tokens.shape)
print("hand_binary shape:", hand_binary.shape)
print("first hand_binary [left, right]:", hand_binary[0])
print("first hand_closed [left, right]:", hand_closed[0])
print("first motion_token[:16]:", np.array2string(motion_tokens[0, :16], precision=4, suppress_small=True))

In [ ]:
def load_ground_truth_chunk(table, start: int, horizon: int) -> np.ndarray:
    end = min(start + horizon, table.num_rows)
    rows = []
    for r in range(start, end):
        mt = np.asarray(row_value(table, "action.motion_token", r), dtype=np.float32)
        hb = np.asarray(row_value(table, "action.hand_binary", r), dtype=np.float32)
        rows.append(np.concatenate([mt, hb]))
    return np.stack(rows, axis=0)

gt_actions = load_ground_truth_chunk(table, frame_index, len(actions))
compare_len = min(len(actions), len(gt_actions))
motion_mae = np.mean(np.abs(motion_tokens[:compare_len] - gt_actions[:compare_len, :MOTION_TOKEN_DIM]))
hand_mae = np.mean(np.abs(hand_binary[:compare_len] - gt_actions[:compare_len, MOTION_TOKEN_DIM:ACTION_DIM]))
first_motion_l1 = np.mean(np.abs(motion_tokens[0] - gt_motion_token))
first_hand_l1 = np.mean(np.abs(hand_binary[0] - gt_hand_binary))

print("Comparison to recorded demo actions from the same frame")
print("compare_len:", compare_len)
print("chunk motion_token MAE:", float(motion_mae))
print("chunk hand_binary MAE:", float(hand_mae))
print("first-step motion_token MAE:", float(first_motion_l1))
print("first-step hand_binary MAE:", float(first_hand_l1))
print("gt first hand_binary:", gt_hand_binary)
print("pred first hand_binary:", hand_binary[0])

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=False)
axes[0].plot(motion_tokens[:, :8])
axes[0].set_title("Predicted motion_token dims 0-7")
axes[0].set_ylabel("value")
axes[1].plot(hand_binary[:, 0], label="left")
axes[1].plot(hand_binary[:, 1], label="right")
axes[1].axhline(HAND_BINARY_CLOSE_THRESHOLD, color="k", linestyle="--", linewidth=1)
axes[1].set_title("Predicted hand_binary_action")
axes[1].legend()
axes[2].imshow(motion_tokens.T, aspect="auto", interpolation="nearest")
axes[2].set_title("Predicted motion_token heatmap [dim x timestep]")
axes[2].set_xlabel("timestep")
axes[2].set_ylabel("motion dim")
plt.tight_layout()
plt.show()

The values above are HEX policy output only. No Sonic command socket is opened, and no `gear_sonic` module is imported.

In [ ]:
client.close()